In [ ]:
# Clone repo and install deps
!git clone https://github.com/michaelyserrano/mmai.git
%cd mmai
!pip install -q -r project/requirements.txt
import sys
sys.path.insert(0, "project")

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
from data.load_toolbench import load_api_corpus
from models.embeddings import format_api_string, get_embeddings
import numpy as np
from tqdm import tqdm
from pathlib import Path

TOOLBENCH_DIR = Path("toolbench")  # mount your Drive or upload data
corpus = load_api_corpus(TOOLBENCH_DIR / "toolenv" / "tools")
print(f"Loaded {len(corpus)} APIs across {len(set(a['category'] for a in corpus))} categories")

In [ ]:
api_strings = [format_api_string(a) for a in corpus]
print("Sample:", api_strings[0])

# Embed all APIs (cached — safe to re-run)
BATCH = 500
all_embeddings = []
for i in tqdm(range(0, len(api_strings), BATCH)):
    batch = api_strings[i:i+BATCH]
    embs = get_embeddings(batch)
    all_embeddings.append(embs)

corpus_matrix = np.vstack(all_embeddings)
np.save("project/corpus_embeddings.npy", corpus_matrix)
print(f"✓ Saved corpus_embeddings.npy: shape {corpus_matrix.shape}")

In [ ]:
import json
api_names = [a["name"] for a in corpus]
with open("project/api_names.json", "w") as f:
    json.dump(api_names, f)
print(f"✓ Saved {len(api_names)} API names")